# Fine-tuning — Dataset Misto e Adapter-Only

Objetivos deste notebook:
- abandonar o pipeline em duas fases
- treinar com dataset misto mais alinhado ao problema
- manter configuração reprodutível e estável no Colab
- salvar apenas o adapter LoRA para uso posterior no sistema


## Instalar dependências

In [ ]:
!pip install -q unsloth transformers datasets accelerate bitsandbytes torch trl peft gdown


## Configuração

In [ ]:
import os
import random
import json
from pathlib import Path

SEED = 42
random.seed(SEED)

BASE_MODEL = "unsloth/Qwen2.5-14B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 512
BATCH_SIZE = 1
GRAD_ACC = 8
LEARNING_RATE = 2e-5
NUM_EPOCHS = 1
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05
OUTPUT_DIR = "models/final_adapter"
OUTPUT_ZIP_NAME = "medical_assistant_llm_adapter.zip"

MEDPT_FIELD = "medical_specialty"
MEDPT_SPECIALTIES = [
    "Cardiologista",
    "Neurologista",
    "Gastroenterologista",
    "Infectologista",
    "Cirurgião geral",
    "Cirurgião do aparelho digestivo, Cirurgião geral",
    "Nefrologista",
    "Pneumologista",
    "Clínico Geral",
]
MEDPT_SAMPLES_PER_SPECIALTY = 250

SYNTHETIC_DRIVE_PATH = "/content/drive/MyDrive/FIAP-TC-Fase3/dataset_clinical_cases_generated.jsonl"
GOLDEN_DRIVE_PATH = "/content/drive/MyDrive/FIAP-TC-Fase3/dataset_golden_cases.jsonl"

TARGET_MEDPT = 2000
TARGET_SYNTHETIC = 1500
TARGET_GOLDEN = 300

print("Configuracao carregada.")
print(json.dumps({
    "BASE_MODEL": BASE_MODEL,
    "MAX_SEQ_LENGTH": MAX_SEQ_LENGTH,
    "BATCH_SIZE": BATCH_SIZE,
    "GRAD_ACC": GRAD_ACC,
    "LEARNING_RATE": LEARNING_RATE,
    "NUM_EPOCHS": NUM_EPOCHS,
    "TARGET_MEDPT": TARGET_MEDPT,
    "TARGET_SYNTHETIC": TARGET_SYNTHETIC,
    "TARGET_GOLDEN": TARGET_GOLDEN,
}, ensure_ascii=False, indent=2))


## Montar Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Carregar modelo base e anexar LoRA

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id
model.config.pad_token_id = tokenizer.pad_token_id

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

print("Modelo pronto com LoRA anexado.")


## Dataset 1 — MedPT filtrado

In [ ]:
from collections import defaultdict, Counter
from datasets import load_dataset

medpt_base = load_dataset("AKCIT/MedPT", split="train")
counts = Counter(medpt_base[MEDPT_FIELD])
print("Especialidades disponíveis:", counts.most_common(20))

selected_indices = []
indices_by_specialty = defaultdict(list)

for idx, row in enumerate(medpt_base):
    specialty = row.get(MEDPT_FIELD)
    if specialty in MEDPT_SPECIALTIES:
        indices_by_specialty[specialty].append(idx)

for specialty, indices in indices_by_specialty.items():
    random.shuffle(indices)
    selected_indices.extend(indices[:MEDPT_SAMPLES_PER_SPECIALTY])

medpt_dataset = medpt_base.select(selected_indices)
print("Registros MedPT selecionados:", len(medpt_dataset))
print("Especialidades realmente usadas:", sorted(set(medpt_dataset[MEDPT_FIELD])))


In [ ]:
def medpt_to_messages(example):
    specialty = example.get("medical_specialty", "Clínico Geral")
    return {
        "messages": [
            {"role": "system", "content": f"Você é um assistente clínico focado em raciocínio médico seguro. Especialidade predominante do contexto: {specialty}. Se faltarem dados, responda como insufficient_data. Se estiver fora do escopo principal, responda como out_of_scope e sugira a especialidade. Nunca invente histórico ou achados ausentes."},
            {"role": "user", "content": example["question"]},
            {"role": "assistant", "content": example["answer"]},
        ]
    }

medpt_dataset = medpt_dataset.map(medpt_to_messages)
medpt_dataset = medpt_dataset.select(range(min(TARGET_MEDPT, len(medpt_dataset))))
print(medpt_dataset[0]["messages"])


## Dataset 2 — Sintético corrigido

In [ ]:
from datasets import load_dataset

if not os.path.exists(SYNTHETIC_DRIVE_PATH):
    raise FileNotFoundError(
        f"Dataset sintético não encontrado em: {SYNTHETIC_DRIVE_PATH}. Rode o colab de geração antes."
    )

synthetic_dataset = load_dataset(
    "json",
    data_files=SYNTHETIC_DRIVE_PATH,
    split="train",
)
synthetic_dataset = synthetic_dataset.select(range(min(TARGET_SYNTHETIC, len(synthetic_dataset))))
print("Registros sintéticos:", len(synthetic_dataset))
print(synthetic_dataset[0]["messages"][-1]["content"][:500])


## Dataset 3 — Golden set opcional

In [ ]:
from datasets import Dataset

if os.path.exists(GOLDEN_DRIVE_PATH):
    golden_dataset = load_dataset(
        "json",
        data_files=GOLDEN_DRIVE_PATH,
        split="train",
    )
    golden_dataset = golden_dataset.select(range(min(TARGET_GOLDEN, len(golden_dataset))))
    print("Registros golden:", len(golden_dataset))
else:
    print("Golden set não encontrado. Seguindo sem essa fatia por enquanto.")
    golden_dataset = Dataset.from_list([])


## Unificar datasets

In [ ]:
from datasets import concatenate_datasets

parts = [medpt_dataset, synthetic_dataset]
if len(golden_dataset) > 0:
    parts.append(golden_dataset)

mixed_dataset = concatenate_datasets(parts).shuffle(seed=SEED)
print("Dataset misto total:", len(mixed_dataset))


In [ ]:
def format_messages(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

mixed_dataset = mixed_dataset.map(format_messages)
mixed_dataset = mixed_dataset.remove_columns([c for c in mixed_dataset.column_names if c != "text"])
print(mixed_dataset[0]["text"][:800])


## Split de treino e avaliação

In [ ]:
split = mixed_dataset.train_test_split(test_size=0.05, seed=SEED)
train_dataset = split["train"]
eval_dataset = split["test"]

print("Train:", len(train_dataset))
print("Eval:", len(eval_dataset))


## Treino

In [ ]:
from transformers import TrainingArguments, EarlyStoppingCallback
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACC,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    logging_steps=10,
    eval_steps=50,
    save_steps=50,
    bf16=True,
    optim="paged_adamw_8bit",
    eval_strategy="steps",
    save_strategy="steps",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    args=training_args,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()


## Salvar adapter-only

In [ ]:
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

required_files = [
    "adapter_config.json",
    "adapter_model.safetensors",
    "tokenizer_config.json",
]
for name in required_files:
    path = Path(OUTPUT_DIR) / name
    print(name, "OK" if path.exists() else "MISSING")


In [ ]:
import zipfile

zip_path = Path("models") / OUTPUT_ZIP_NAME
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for file_path in Path(OUTPUT_DIR).rglob("*"):
        if file_path.is_file():
            zf.write(file_path, arcname=file_path.relative_to(OUTPUT_DIR))

print(f"Adapter salvo em: {zip_path}")


## Copiar para o Drive

In [ ]:
target_dir = Path('/content/drive/MyDrive/FIAP-TC-Fase3')
target_dir.mkdir(parents=True, exist_ok=True)
!cp "models/{OUTPUT_ZIP_NAME}" "{target_dir}/"
print(f"Arquivo copiado para: {target_dir / OUTPUT_ZIP_NAME}")


## Observações finais

Este notebook foi simplificado para priorizar:
- reprodutibilidade
- estabilidade operacional
- dataset misto coerente com o objetivo clínico
- save adapter-only
